# 01 — Data Cleaning

This notebook takes the raw Online Retail II export and turns it into a clean, analysis-ready dataset. **Input:** `data/raw/online_retail_II.csv` (541,910 line items, Dec 2010–Dec 2011, UK-based online retailer). It resolves data quality issues (duplicates, bad-debt adjustments, internal stock write-offs, missing values, dtype mismatches), flags cancelled and zero-value orders, and exports two processed datasets — `revenue_df.csv` (full transaction set) and `customer_df.csv` (customer-attributed subset) — that every notebook downstream builds on.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding='ISO-8859-1')

In [3]:
df.shape

(541910, 8)

In [4]:
df.describe()

,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  str    
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [6]:
df.sample(100)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
230964,557225,20727,LUNCH BAG BLACK SKULL.,1,6/17/11 13:37,1.65,15311.0,United Kingdom
13405,537434,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/6/10 16:57,2.51,NaN,United Kingdom
281112,561513,22727,ALARM CLOCK BAKELIKE RED,1,7/27/11 15:12,7.46,NaN,United Kingdom
62417,541497,22031,BOTANICAL LAVENDER BIRTHDAY CARD,2,1/18/11 15:19,0.42,NaN,United Kingdom
184719,552702,22773,GREEN DRAWER KNOB ACRYLIC EDWARDIAN,2,5/10/11 16:00,2.46,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
301619,563349,84692,BOX OF 24 COCKTAIL PARASOLS,1,8/15/11 13:51,0.42,14623.0,United Kingdom
437808,574298,16237,SLEEPING CAT ERASERS,4,11/3/11 15:56,0.42,NaN,United Kingdom
441254,574561,23081,GREEN METAL BOX ARMY SUPPLIES,1,11/4/11 15:52,16.63,NaN,United Kingdom
40821,539743,22501,PICNIC BASKET WICKER LARGE,1,12/21/10 15:20,21.23,NaN,United Kingdom


→ Structurally sound UK retail line items; some rows have very low Price, worth a closer look. (Followed up below at "Zero-value rows" — the Price = 0 case.)

In [7]:
df[~df['Invoice'].str.isnumeric()]['Invoice'].str[0].value_counts()

Invoice
C    9288
A       3
Name: count, dtype: int64

In [8]:
df[df['Invoice'].str.startswith('C', na=False)]['Quantity'].describe()

count     9288.000000
mean       -29.885228
std       1145.786965
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

→ C-invoices always have negative quantity — prefix and sign agree, confirms cancellations.

In [9]:
non_c_returns = df[(df['Quantity'] < 0) & (df['Price'] == 0) & (~df['Invoice'].str.startswith('C', na=False)) & (~df['Invoice'].str.startswith('A', na=False))]
print(non_c_returns['Price'].eq(0).all())
print(non_c_returns['Customer ID'].isna().all())
non_c_returns

True
True


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/10 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/10 15:30,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,12/7/11 18:36,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,12/7/11 18:38,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,12/7/11 18:38,0.0,NaN,United Kingdom
536910,581226,23090,missing,-338,12/8/11 9:56,0.0,NaN,United Kingdom


In [10]:
a_invoices = df[df['Invoice'].str.startswith('A', na=False)]

In [11]:
df.isna().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [12]:
df.duplicated().sum()

5268

In [13]:
df.nunique()

Invoice        25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
Price           1630
Customer ID     4372
Country           38
dtype: int64

## Summary of issues found

1 - invoice date is str it should become datetime\
2 - customer id is float while we don't need floating points\
3 - customerId is null for 135,080 rows — **resolved:** kept, not filled/dropped — `revenue_df` retains these (real guest-checkout revenue), `customer_df` excludes them via `dropna` at export\
4 - There are negative values for price --> return invoices — **resolved:** both negative-Price rows are 'A'-prefixed bad-debt invoices, removed by the 'A'-invoice drop\
5 - Description is null for 1,454 rows\
6 - Quantity has negative values (min -80995), likely returns/cancellations or stock adjustments\
7 - There are 5,268 duplicate rows \
8 - There are 4070 Stock code while 4223 Description which suggests some StockCodes map to multiple descriptions (typos/inconsistent naming) — **deferred:** ~352 StockCodes affected, <0.1% of rows, doesn't touch `customer_df`/RFM (none of the junk-description rows have a Customer ID). Not worth fixing now; revisit if a future notebook (e.g. product co-occurrence) actually needs clean Description-based grouping.
9 - There are 1,179 rows with Price = 0 and Quantity > 0 (not caught by the existing negative-quantity write-off filter) — 1,139 are internal notes with no Customer ID, 40 are real customer orders receiving a free/promotional item

## Cleaning steps applied
1. Dropped exact duplicates
2. Dropped 'A' (bad-debt) invoices
3. Dropped 1,336 `non_c_returns` write-offs (negative-quantity internal adjustments, issue #6)
4. Converted InvoiceDate → datetime, Customer ID → int
5. Flagged cancellations (`is_cancelled`)
6. Filled missing Description with 'Unknown'
7. Added `line_revenue` (Quantity × Price)
8. Dropped 1,139 Price=0/Quantity>0 rows with no Customer ID (same internal-adjustment pattern as the write-offs above)
9. Flagged the remaining 40 Price=0/Quantity>0 rows with `is_zero_value` (real customer orders, kept but excluded from order/frequency counts downstream)

In [14]:
df = df.drop_duplicates()

5268 rows dropped

In [15]:
df = df.drop(a_invoices.index)

bad debt adjustments

In [16]:
df = df.drop(non_c_returns.index)

### Zero-value rows (Price = 0, Quantity > 0)
Positive-quantity side of the same issue as `non_c_returns` above. No Customer ID → internal note, drop. Has a Customer ID → real free item on a real order, keep and flag.

In [17]:
zero_qty_pos = df[(df['Price'] == 0) & (df['Quantity'] > 0)]
zero_no_cust = zero_qty_pos[zero_qty_pos['Customer ID'].isna()]
len(zero_qty_pos), len(zero_no_cust)

(1174, 1134)

In [18]:
df = df.drop(zero_no_cust.index)

removed the invocies which were not canclelation were just adjustments or stock

In [19]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%y %H:%M') # keep full datetime (hour/minute) for hourly-level analysis

In [20]:
df['Customer ID'] = df['Customer ID'].astype('Int64')

→ Int64 (nullable) instead of int64 to keep the remaining nulls without erroring.

In [21]:
df['is_cancelled'] = df['Invoice'].str.startswith('C', na=False)

In [22]:
df['is_zero_value'] = (df['Price'] == 0) & (df['Quantity'] > 0)
df['is_zero_value'].sum()

40

In [23]:
df['Description'] = df['Description'].fillna('Unknown')

In [24]:
df.isna().sum()


Invoice               0
StockCode             0
Description           0
Quantity              0
InvoiceDate           0
Price                 0
Customer ID      132564
Country               0
is_cancelled          0
is_zero_value         0
dtype: int64

In [25]:
df.duplicated().sum()

0

No missing Description, no duplicates. Remaining Customer ID nulls are by design (guest orders, kept for revenue-level analysis).

### Line Revenue
`Quantity × Price` per row — computed once here so it's already in both processed CSVs, instead of recreating it in every downstream notebook.

In [26]:
df['line_revenue'] = df['Quantity'] * df['Price']

In [27]:
revenue_df = df.copy()
customer_df = df.dropna(subset=['Customer ID'])

revenue_df.to_csv('../data/processed/revenue_df.csv', index=False)
customer_df.to_csv('../data/processed/customer_df.csv', index=False)

### Non-Product StockCodes (reference)
Admin/fee codes, not product SKUs — not dropped (real revenue/fees, belong in Gross/Net Sales/AOV as-is). Documented here for future product-level analysis (e.g. co-occurrence) to import and exclude.

In [28]:
# verified individually against raw data — postage, manual adj., discount, samples, bank charges, Amazon fee, charity commission
NON_PRODUCT_CODES = ['POST', 'DOT', 'M', 'D', 'S', 'BANK CHARGES', 'AMAZONFEE', 'CRUK']

## Takeaways

The raw export needed real cleanup before it could support any downstream analysis: 5,268 exact duplicate rows (~1%), 3 bad-debt adjustment invoices, and 1,336 internal stock write-offs (negative-quantity, Price = 0, no Customer ID) all had to be identified and removed — none represent real customer transactions, and leaving them in would have inflated revenue and return figures.

A second, related issue surfaced separately: 1,174 rows with Price = 0 but *positive* Quantity — the write-off filter above only catches the negative-quantity side. 1,134 of these have no Customer ID (same internal-adjustment pattern, dropped); the remaining 40 have a real Customer ID — genuine free/promotional items on real orders, so they're kept but flagged `is_zero_value` so order/frequency counts downstream don't mistake them for a real purchase.

The remaining ~25% of rows with no Customer ID aren't a data quality problem — they're guest checkouts, real revenue that just can't be attributed to a specific customer. That's why the pipeline exports two files rather than one: `revenue_df` (everything, for revenue-level KPIs) and `customer_df` (customer-attributed only, for anything requiring a Customer ID).

Cancelled orders (`is_cancelled`, ~9,288 invoices) were flagged, not dropped, so downstream notebooks can compute Net Sales and return rate rather than losing that signal entirely.